# FUSE-CDR: HGT depth study + pathway sharding

This notebook runs both controlled experiments on a Colab **T4 GPU**. It first runs regression and protocol checks, then executes:

1. The strict HGT study: `local_only`, `hgt_2_only`, `hgt_2`, and `hgt_3`, with seed 0 and five folds.
2. The Omics Encoder pathway-sharding study: 1, 2, 4, 8, and 16 shards, with seed 0 and five folds.

The HGT variants all use the strict FUSE-CDR runner and each dataset's selected random-protocol hyperparameters. The sharding variants all use the same flexible-model trainer and defaults as the neighboring Omics Encoder experiments. Completed folds are reused, so interrupted runs can be resumed.


In [ ]:
import importlib.util
import subprocess
import sys

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()
if "T4" not in gpu_name:
    raise RuntimeError(
        f"This notebook is configured for a Colab T4, but found: {gpu_name!r}. "
        "Select Runtime > Change runtime type > T4 GPU and reconnect."
    )

missing_packages = []
for module_name, package in (
    ("hickle", "hickle==5.0.2"),
    ("torch_geometric", "torch-geometric==2.6.1"),
):
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package)
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

import torch
import torch_geometric

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access the Colab GPU.")
print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__} | CUDA runtime: {torch.version.cuda}")
print(f"PyG: {torch_geometric.__version__}")


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount("/content/drive")

# Change only this path if the repository is stored elsewhere in Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CDRP models testing").resolve()
if not (PROJECT_ROOT / "3OmicsStrictBenchmarking").is_dir():
    raise FileNotFoundError(f"Repository not found at {PROJECT_ROOT}")

HGT_DATASETS = ["dataset-1", "dataset-2"]
HGT_VARIANTS = ["local_only", "hgt_2_only", "hgt_2", "hgt_3"]
FOLD_IDS = [1, 2, 3, 4, 5]
HGT_EPOCHS = 400
SHARD_COUNTS = [1, 2, 4, 8, 16]
SHARD_EPOCHS = 120
RUN_TAG = "verified_t4_v1"
HGT_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "3OmicsStrictBenchmarking"
    / "results"
    / "hgt_depth_study_historical_v1"
)

def run_command(parts):
    command = [str(part) for part in parts]
    print("$", subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print(f"Project: {PROJECT_ROOT}")
print(f"HGT output: {HGT_OUTPUT_ROOT}")
print(f"Sharding tag: {RUN_TAG}")


## Regression checks

These tests verify that explicitly selecting two local and two HGT layers is numerically identical to the legacy `num_layers=2` full model, and that the corrected one-shard pathway input is numerically identical to the original unsplit pathway-only baseline.


In [ ]:
run_command(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_*.py",
        "-v",
    ]
)


## Protocol checks

The HGT preflight reads the canonical strict random splits directly, verifies that they exactly reproduce the historical split algorithm, checks disjoint and complete train/validation/test partitions, and requires every test assignment to match all available saved benchmark prediction files.

The sharding preflight reconstructs the original pathway matrix from every shard configuration and checks that Experiments 1 through 4 use the same seed, epoch, and fold defaults.


In [ ]:
import importlib.util
import json
from unittest import mock

import pandas as pd

for path in (PROJECT_ROOT, PROJECT_ROOT / "flexible model"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from benchmark_wrappers.fusecdr_hgt_depth_study import (
    EXPERIMENT_FOLDS,
    EXPERIMENT_VARIANTS,
    VARIANT_GLOBAL_LAYERS,
    VARIANT_LOCAL_LAYERS,
    ensure_study_split,
    load_depth_config,
)
from flexibility_utils import build_pathway_shards

assert tuple(FOLD_IDS) == EXPERIMENT_FOLDS
assert tuple(HGT_VARIANTS) == EXPERIMENT_VARIANTS
assert {
    variant: (VARIANT_LOCAL_LAYERS[variant], VARIANT_GLOBAL_LAYERS[variant])
    for variant in HGT_VARIANTS
} == {
    "local_only": (2, 0),
    "hgt_2_only": (0, 2),
    "hgt_2": (2, 2),
    "hgt_3": (2, 3),
}

benchmark_dir = PROJECT_ROOT / "3OmicsStrictBenchmarking"
for dataset in HGT_DATASETS:
    split_dir = ensure_study_split(
        benchmark_dir=str(benchmark_dir),
        output_root=str(HGT_OUTPUT_ROOT),
        dataset=dataset,
        fold_ids=FOLD_IDS,
    )
    audit_path = HGT_OUTPUT_ROOT / "split_audits" / f"{dataset}.json"
    audit = json.loads(audit_path.read_text())
    if not all(
        row["saved_prediction_matches"]
        and all(row["saved_prediction_matches"].values())
        for row in audit["folds"]
    ):
        raise RuntimeError(f"{dataset}: strict saved test assignments could not be verified")
    config = load_depth_config(str(benchmark_dir), dataset)
    print(f"{dataset}: {audit['prepared_response_pairs']:,} pairs; strict test folds verified")
    print("  canonical split:", split_dir)
    print("  selected strict config:", config)

def load_script(filename, module_name):
    path = PROJECT_ROOT / "flexible model" / filename
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

protocol_defaults = {}
for filename in (
    "run_exp1_subset_cardinality.py",
    "run_exp2_pathway_sharding.py",
    "run_exp3_missing_modality.py",
    "run_exp4_noisy_modality.py",
):
    module = load_script(filename, f"audit_{Path(filename).stem}")
    with mock.patch.object(sys, "argv", [filename]):
        args = module.parse_args()
    protocol_defaults[filename] = (args.seed, args.epochs, args.k_fold)
if set(protocol_defaults.values()) != {(0, 120, 5)}:
    raise RuntimeError(f"Neighboring Omics Encoder protocols differ: {protocol_defaults}")
print("Omics Encoder defaults (seed, epochs, folds):", protocol_defaults)

pathway = pd.read_csv(PROJECT_ROOT / "final_dataset" / "pathway.csv", index_col=0)
for count in SHARD_COUNTS:
    shards = build_pathway_shards(
        base_dataset_root=PROJECT_ROOT / "final_dataset",
        shard_count=count,
    )
    reconstructed = pd.concat([shards[name] for name in sorted(shards)], axis=1)
    pd.testing.assert_frame_equal(reconstructed, pathway)
    print(f"{count:>2} shard(s): widths={[frame.shape[1] for frame in shards.values()]}")


## Run the HGT study

This is the long-running section: 2 datasets x 4 variants x 5 folds = 40 fold models. The propagation probe is run on dataset-2.


In [ ]:
run_command(
    [
        sys.executable,
        PROJECT_ROOT / "3OmicsStrictBenchmarking" / "run_hgt_depth_study.py",
        "--datasets",
        *HGT_DATASETS,
        "--variants",
        *HGT_VARIANTS,
        "--fold-ids",
        *FOLD_IDS,
        "--device",
        "cuda",
        "--epochs",
        HGT_EPOCHS,
        "--probe-dataset",
        "dataset-2",
        "--output-root",
        HGT_OUTPUT_ROOT,
    ]
)


In [ ]:
from IPython.display import display
import pandas as pd

for filename in (
    "variant_summary.csv",
    "distance_summary.csv",
    "propagation_summary.csv",
):
    path = HGT_OUTPUT_ROOT / filename
    if path.is_file():
        print(filename)
        display(pd.read_csv(path))


## Run pathway sharding

Each view contains the same response, graph, similarity, and physicochemical inputs. Only the original pathway columns are divided into the requested number of pathway subtype tensors.


In [ ]:
run_command(
    [
        sys.executable,
        PROJECT_ROOT / "flexible model" / "run_exp2_pathway_sharding.py",
        "--dataset-root",
        PROJECT_ROOT / "final_dataset",
        "--device",
        "cuda",
        "--seed",
        0,
        "--epochs",
        SHARD_EPOCHS,
        "--k-fold",
        5,
        "--shard-counts",
        *SHARD_COUNTS,
        "--tag",
        RUN_TAG,
    ]
)


In [ ]:
from datetime import datetime, timezone
import json
from IPython.display import display
import pandas as pd

sharding_output = (
    PROJECT_ROOT
    / "flexible model"
    / "flexibility_outputs"
    / "exp2_pathway_sharding"
    / RUN_TAG
)
sharding_results = pd.read_csv(sharding_output / "results.csv")
if sharding_results["shard_count"].tolist() != SHARD_COUNTS:
    raise RuntimeError("Pathway-sharding output is incomplete or out of order.")
if not sharding_results["shard_stems"].str.startswith("pathway_shard_01").all():
    raise RuntimeError("The corrected explicit pathway shard selectors were not recorded.")
display(sharding_results)

combined_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": gpu_name,
    "seed": 0,
    "folds": FOLD_IDS,
    "hgt": {
        "datasets": HGT_DATASETS,
        "variants": HGT_VARIANTS,
        "epochs": HGT_EPOCHS,
        "output_root": str(HGT_OUTPUT_ROOT),
    },
    "pathway_sharding": {
        "shard_counts": SHARD_COUNTS,
        "epochs": SHARD_EPOCHS,
        "output_root": str(sharding_output),
    },
}
manifest_path = PROJECT_ROOT / f"combined_experiment_{RUN_TAG}.json"
manifest_path.write_text(json.dumps(combined_manifest, indent=2) + "\n")
print(f"Combined manifest: {manifest_path}")
